# Отчет по работе

## Методология

1. Загрузить данные. Данные хранятся в `data.csv` - 100 рядов из
M4 daily, предварительно загружены (так как были конфликты с версией numpy при
загрузке в процессе выполнения файлы)
2. Разделить данные на train и test - в test отправится 14 значений горизонта
3. Подготовить функции для работы с моделями:
В качестве baseline используются Naive, SeasonalNaive, AutoTheta и AutoETS.
CatBoost и PatchTST используются как основные модели.
Функции, соответственно, получают на вход обучающую выборку и горизонт
для предсказаний. Бейзлайны возвращаюсь сразу датасет с результатами всех 4 моделей, CatBoost и PatchTST
возвращают датасет с `unique_id` (id ряда), `ds` (номер дня), model (название модели, столбец с предсказаниями). Лаги для CatBoost считаются после скейлинга.
4. Подготовить scalers: обучить по скейлеру на каждый ряд исключительно на обучающей выборке и сделать функции
трансформации и обратной трансформации для датасета формата `unique_id`, `ds`, `y`.
5. Порядок экспериментов: для каждого скейлера (оригинальный `y` получаем через `scaler=None`) получаем
предсказания от всех 6 моделей. Считаем метрику MAE (базовый вариант для временных рядов, нам важно посмотреть
на изменения) на тестовых данных (сделав обратную трансформацию для предсказаний). Значения метрик сохраням в датафрейм.
6. Анализируем эксперимент на основе таблицы с метриками `mae.csv`

## Анализ метрик

In [1]:
import pandas as pd

df = pd.read_csv("mae.csv")
df

,Unnamed: 0,Original,Standard,Robust,Quantile
0,Naive,102.376645,102.376645,1.023766e+02,102.376645
1,SeasonalNaive,138.435240,138.435240,1.384352e+02,138.435240
2,AutoTheta,107.355732,106.084803,1.057053e+02,108.555842
3,AutoETS,395.175453,108.138801,1.081007e+02,102.201508
4,CatBoost,117.326116,173.885046,6.630484e+06,166.705767
5,PatchTST,103.141100,103.513332,1.049971e+02,106.274023


- Поведение бейзлайнов Naive, SeasonalNaive, AutoTheta ожидаемо - значения MAE практически не меняются (+- 1). Скейлинг не должен влиять на эти бейзлайны, так как мы возвращаемся к исходному масштабу предсказаний (модели линейные).
- С AutoETS довольно неожиданный результат (подозрительно высокое значение на оригинальных данных). Заподозрила ошибку при вычислениях, но не смогла найти.
- CatBoost со скейлингами Standard и Quantile работает немного хуже, чем с оригинальными данными - возможно данные сжимаются и пороги для деревьев выбираются чуть иначе. Также quantile в целом нелинейный, ломает структуру.
- Есть аномальное значение для CatBoost+RobustScaler - слишком большое. Также подозреваю ошибку в вычислениях, но не смогла отловить (используются одинаковые функции для всех моделей и скейлеров)
- В PatchTST ожидала скорее существенного улучшения, но оно не наблюдается. Возможно, на других гиперпараметрах модели разница бы появилась. Либо стоило сделать скейлинг сразу по всем рядам (а не один scaler на один ряд) - чтобы сохранилась общая структура рядов.